<a href="https://colab.research.google.com/github/V1LL4pro/Deep-Learning/blob/Corte-1/T3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import os
from PIL import Image
from tqdm import tqdm
import requests
import urllib.parse

# Configuración inicial
BASE_PATH = "/content/quickdraw_dataset"
MAX_IMAGES_PER_CLASS = 1000  # Ajustable según necesidades

# Obtener la lista completa de categorías
categories_url = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
response = requests.get(categories_url)
original_categories = [line.strip() for line in response.text.split('\n') if line.strip()]

# Crear estructura de directorios con nombres válidos
os.makedirs(BASE_PATH, exist_ok=True)
for category in original_categories:
    safe_name = category.replace(" ", "_")
    os.makedirs(os.path.join(BASE_PATH, safe_name), exist_ok=True)

def process_category(original_name):
    try:
        # Generar nombres seguros para URL y sistema de archivos
        url_name = urllib.parse.quote(original_name)
        dir_name = original_name.replace(" ", "_")

        # Descargar el archivo .npy
        url = f"https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{url_name}.npy"
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Guardar temporalmente
        temp_path = os.path.join(BASE_PATH, f"{dir_name}.npy")
        with open(temp_path, "wb") as f:
            f.write(response.content)

        # Procesar imágenes
        images = np.load(temp_path)
        np.random.shuffle(images)

        # Guardar imágenes en formato PNG
        for i, img_data in enumerate(images[:MAX_IMAGES_PER_CLASS]):
            img = Image.fromarray(img_data.reshape(28, 28).astype(np.uint8))
            img.save(os.path.join(BASE_PATH, dir_name, f"{dir_name}_{i}.png"))

        # Limpiar archivo temporal
        os.remove(temp_path)

    except Exception as e:
        print(f"Error en {original_name}: {str(e)}")

# Procesar todas las categorías
for category in tqdm(original_categories):
    process_category(category)

print("Proceso completado exitosamente!")

 43%|████▎     | 147/345 [03:53<05:36,  1.70s/it]